In [1]:
import numpy as np
from qiskit.quantum_info import partial_trace, entropy
from qiskit.quantum_info import Statevector

# Photon operators (3-level truncation)
a = np.array([[0,1,0],
              [0,0,np.sqrt(2)],
              [0,0,0]])   # annihilation
adag = a.T                # creation
n_photon = adag @ a        # number operator

# Atomic operators (2-level system)
sigma_z = np.array([[1,0],[0,-1]])
sigma_plus = np.array([[0,1],[0,0]])
sigma_minus = np.array([[0,0],[1,0]])



In [ ]:
from numpy import kron

# Identity operators
I3 = np.eye(3)   # photon identity
I2 = np.eye(2)   # atom identity
I6 = np.eye(6)   # site identity

# Site operators
a_site = kron(a, I2)
adag_site = kron(adag, I2)
n_site = kron(n_photon, I2)

sigma_z_site = kron(I3, sigma_z)
sigma_plus_site = kron(I3, sigma_plus)
sigma_minus_site = kron(I3, sigma_minus)

# Operators for site 1 and site 2
a1 = kron(a_site, I6)
a2 = kron(I6, a_site)

sigma_z1 = kron(sigma_z_site, I6)
sigma_z2 = kron(I6, sigma_z_site)

# Build the spin-ladder operators for each site
sigma_minus1 = kron(sigma_minus_site, I6)
sigma_plus1 = kron(sigma_plus_site, I6)
sigma_minus2 = kron(I6, sigma_minus_site)
sigma_plus2 = kron(I6, sigma_plus_site)

# Parameters
wc = 1.0   # cavity frequency
wa = 1.0   # atom frequency
g  = 0.1   # coupling
J  = 0.05  # hopping

# JC Hamiltonian for each site
H_JC1 = wc * (a1.T @ a1) + (wa/2)*sigma_z1 + g*(a1.T @ sigma_minus1 + a1 @ sigma_plus1)
H_JC2 = wc * (a2.T @ a2) + (wa/2)*sigma_z2 + g*(a2.T @ sigma_minus2 + a2 @ sigma_plus2)

# Hopping term
H_hop = -J * (a1.T @ a2 + a2.T @ a1)

# Total Hamiltonian
H = H_JC1 + H_JC2 + H_hop


eigvals, eigvecs = np.linalg.eigh(H)
ground_state = eigvecs[:,0]   # lowest energy eigenvector
psi = Statevector(ground_state)


rho_site1 = partial_trace(psi, [1])   # trace out site 2
S = entropy(rho_site1, base=2)        # von Neumann entropy
print("Entanglement entropy:", S)


NameError: name 'sigma_minus1' is not defined

In [ ]:
import matplotlib.pyplot as plt

J_values = np.linspace(0,0.5,20)
entropies = []

for J in J_values:
    H = H_JC1 + H_JC2 - J*(a1.T @ a2 + a2.T @ a1)
    eigvals, eigvecs = np.linalg.eigh(H)
    psi = Statevector(eigvecs[:,0])
    rho_site1 = partial_trace(psi, [1])
    entropies.append(entropy(rho_site1, base=2))

plt.plot(J_values, entropies)
plt.xlabel("Hopping J")
plt.ylabel("Entanglement Entropy")
plt.show()
